# M1.0a Primer: Vectors and Fields

Book pointer: chapter 1, sections 1.1 (vectors) and 1.2 (multivariate functions and vector fields).

This is the first of four primer notebooks (00a to 00d) that walk through chapter 1 slowly. They build every idea from a picture or a measurement before writing the formula, do the algebra by hand in worked examples, and then reproduce each number in plain NumPy so nothing is hidden. If the warm-up questions are easy, skip to `01_grids_and_operators`, which is the compact version of all four.

## 1. Warm-up

Try on paper first.

1. What is the difference between a vector and its components? Why can the same vector have different components?
2. $\mathbf{a}\cdot\mathbf{b}$ and $\mathbf{a}\times\mathbf{b}$: which is a number, which is a vector, and what does each measure?
3. What is the unit vector in the direction of $(3, 4, 0)$?
4. A scalar field and a vector field: give one physical example of each and say what a picture of each looks like.
5. In Coulomb's law the vector from the source charge to the observation point appears. Write it in terms of the two position vectors.

## 2. The idea in one paragraph

Electromagnetism is written in the language of vectors because forces and fields have direction. Section 1.1 gives the arithmetic of single vectors: adding, scaling, and the two products. Section 1.2 extends this to functions of position, scalar fields and vector fields, which is what the electric and magnetic fields are. This notebook covers both, with the computational idiom for each: a vector is a NumPy array of components, a field is an array of values on a grid of points.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 What a vector is (section 1.1)

A vector has a length and a direction. That is the geometric object. To calculate with it we choose three perpendicular reference directions $\hat{\mathbf{x}}, \hat{\mathbf{y}}, \hat{\mathbf{z}}$ (unit vectors, length one) and write the vector as a sum of multiples of them:

$$\mathbf{a} = a_x\hat{\mathbf{x}} + a_y\hat{\mathbf{y}} + a_z\hat{\mathbf{z}}, \qquad \text{written for short as } (a_x, a_y, a_z).$$

The three numbers are the components. They depend on the reference directions: rotate the axes and the same arrow gets different components. The length does not depend on the axes:

$$|\mathbf{a}| = \sqrt{a_x^2 + a_y^2 + a_z^2}.$$

Two operations are built in. Adding vectors adds components (geometrically, place them head to tail). Multiplying by a number scales the length (and flips the direction if the number is negative). Dividing a vector by its own length gives the unit vector $\hat{\mathbf{a}} = \mathbf{a}/|\mathbf{a}|$, which keeps the direction and throws away the length.

**Worked example 1.** $\mathbf{a} = (3, 4, 0)$. Length $|\mathbf{a}| = \sqrt{9 + 16} = 5$. Unit vector $\hat{\mathbf{a}} = (3, 4, 0)/5 = (0.6, 0.8, 0)$, and indeed $0.6^2 + 0.8^2 = 1$. Adding $\mathbf{b} = (1, -1, 2)$ gives $\mathbf{a} + \mathbf{b} = (4, 3, 2)$. Scaling by $-2$ gives $-2\mathbf{a} = (-6, -8, 0)$, the same line, opposite direction, twice as long.

In [ ]:
a = np.array([3.0, 4.0, 0.0])
b = np.array([1.0, -1.0, 2.0])

length = np.linalg.norm(a)                    # sqrt(a @ a)
a_hat = a / length
print("length of a:", length, "   unit vector:", a_hat, "   its length:", np.linalg.norm(a_hat))
print("a + b =", a + b, "    -2 a =", -2 * a)

# A picture of head-to-tail addition in the plane.
fig, ax = plt.subplots(figsize=(5.5, 5))
def arrow(start, vec, colour, label):
    ax.annotate("", xy=(start + vec)[:2], xytext=start[:2], arrowprops=dict(arrowstyle="->", color=colour, lw=2))
    ax.text(*(start + vec / 2)[:2] + np.array([0.1, 0.1]), label, color=colour)
arrow(np.zeros(3), a, "C0", "a")
arrow(a, b, "C1", "b (placed at the head of a)")
arrow(np.zeros(3), a + b, "C2", "a + b")
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("vector addition is head to tail")
plt.show()

### 3.2 The dot product

The dot product turns two vectors into a number. It has three equivalent faces, and the skill is to switch between them freely.

1. Components: $\mathbf{a}\cdot\mathbf{b} = a_x b_x + a_y b_y + a_z b_z$. This is how you compute it.
2. Geometry: $\mathbf{a}\cdot\mathbf{b} = |\mathbf{a}||\mathbf{b}|\cos\theta$, with $\theta$ the angle between them. This is how you interpret it: it is zero when they are perpendicular, largest when parallel, negative when they point against each other.
3. Projection: $\mathbf{a}\cdot\hat{\mathbf{b}}$ is the length of the shadow of $\mathbf{a}$ on the line of $\mathbf{b}$. So $(\mathbf{a}\cdot\hat{\mathbf{b}})\,\hat{\mathbf{b}}$ is the part of $\mathbf{a}$ along $\mathbf{b}$, and what is left over, $\mathbf{a} - (\mathbf{a}\cdot\hat{\mathbf{b}})\hat{\mathbf{b}}$, is perpendicular to $\mathbf{b}$.

Properties worth knowing by heart: it is symmetric, $\mathbf{a}\cdot\mathbf{b} = \mathbf{b}\cdot\mathbf{a}$; it distributes over addition; and $\mathbf{a}\cdot\mathbf{a} = |\mathbf{a}|^2$. The unit vectors satisfy $\hat{\mathbf{x}}\cdot\hat{\mathbf{x}} = 1$ and $\hat{\mathbf{x}}\cdot\hat{\mathbf{y}} = 0$, which is where face 1 comes from when you expand $(a_x\hat{\mathbf{x}} + \dots)\cdot(b_x\hat{\mathbf{x}} + \dots)$.

Physics uses the dot product whenever only the part of one vector along another matters. Work is $\mathbf{F}\cdot\mathbf{d}$, force along displacement. Flux is $\mathbf{E}\cdot\hat{\mathbf{n}}$, field along the surface normal. Both are everywhere in this course.

**Worked example 2.** $\mathbf{a} = (3, 1, 0)$, $\mathbf{b} = (1, 2, 0)$. Face 1: $\mathbf{a}\cdot\mathbf{b} = 3 + 2 = 5$. Lengths $\sqrt{10}$ and $\sqrt5$. Face 2: $\cos\theta = 5/\sqrt{50} = 1/\sqrt2$, so $\theta = 45^\circ$. Face 3: $\hat{\mathbf{b}} = (1, 2, 0)/\sqrt5$, shadow length $\mathbf{a}\cdot\hat{\mathbf{b}} = 5/\sqrt5 = \sqrt5 = 2.236$, part of $\mathbf{a}$ along $\mathbf{b}$ is $\sqrt5\,\hat{\mathbf{b}} = (1, 2, 0)$, and the perpendicular remainder is $(3, 1, 0) - (1, 2, 0) = (2, -1, 0)$. Check: $(2, -1, 0)\cdot(1, 2, 0) = 0$.

**Worked example 3.** Is $(1, 2, 3)$ perpendicular to $(3, 0, -1)$? Dot product $3 + 0 - 3 = 0$. Yes. What angle does $(1, 1, 1)$ make with the $z$ axis? $\cos\theta = (1,1,1)\cdot(0,0,1)/(\sqrt3\cdot1) = 1/\sqrt3$, $\theta = 54.7^\circ$.

In [ ]:
a = np.array([3.0, 1.0, 0.0]); b = np.array([1.0, 2.0, 0.0])
dot = a @ b
theta = np.degrees(np.arccos(dot / (np.linalg.norm(a) * np.linalg.norm(b))))
b_hat = b / np.linalg.norm(b)
shadow = a @ b_hat
a_along = shadow * b_hat
a_perp = a - a_along
print(f"a . b = {dot}   angle = {theta:.1f} deg   shadow of a on b = {shadow:.3f}")
print(f"part of a along b = {a_along}   perpendicular remainder = {a_perp}   check remainder . b = {a_perp @ b}")
print(f"(1,2,3) . (3,0,-1) = {np.array([1, 2, 3]) @ np.array([3, 0, -1])}    angle of (1,1,1) with z axis = {np.degrees(np.arccos(1 / np.sqrt(3))):.1f} deg")

fig, ax = plt.subplots(figsize=(5, 5))
for v, c, lbl in ((a, "C0", "a"), (b, "C1", "b"), (a_along, "C2", "along b"), (a_perp, "C3", "perpendicular to b")):
    ax.annotate("", xy=v[:2], xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=2)); ax.text(*v[:2] * 1.06, lbl, color=c)
ax.plot([a_along[0], a[0]], [a_along[1], a[1]], "k:", lw=1)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-1.5, 2.5); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("the dot product splits a into along-b and across-b")
plt.show()

### 3.3 The cross product

The cross product turns two vectors into a third vector that is perpendicular to both. Its length is the area of the parallelogram the two span, $|\mathbf{a}\times\mathbf{b}| = |\mathbf{a}||\mathbf{b}|\sin\theta$, and its direction follows the right-hand rule: curl the fingers of the right hand from $\mathbf{a}$ towards $\mathbf{b}$ and the thumb gives $\mathbf{a}\times\mathbf{b}$.

In components,
$$\mathbf{a}\times\mathbf{b} = (a_y b_z - a_z b_y)\,\hat{\mathbf{x}} + (a_z b_x - a_x b_z)\,\hat{\mathbf{y}} + (a_x b_y - a_y b_x)\,\hat{\mathbf{z}}.$$

A way to remember it: each component is built from the other two indices in cyclic order ($x \to yz$, $y \to zx$, $z \to xy$), as "first times second minus second times first". The determinant with $\hat{\mathbf{x}}, \hat{\mathbf{y}}, \hat{\mathbf{z}}$ in the top row gives the same thing.

Properties: it is antisymmetric, $\mathbf{b}\times\mathbf{a} = -\mathbf{a}\times\mathbf{b}$; a vector crossed with itself (or with anything parallel to it) is zero; and for the unit vectors $\hat{\mathbf{x}}\times\hat{\mathbf{y}} = \hat{\mathbf{z}}$, $\hat{\mathbf{y}}\times\hat{\mathbf{z}} = \hat{\mathbf{x}}$, $\hat{\mathbf{z}}\times\hat{\mathbf{x}} = \hat{\mathbf{y}}$ (cyclic), with a minus sign if the order is reversed.

Physics uses it whenever a result is perpendicular to two inputs. The magnetic force on a moving charge is $q\mathbf{v}\times\mathbf{B}$ (chapter 11). Torque is $\mathbf{r}\times\mathbf{F}$. The Biot-Savart law has $d\mathbf{l}\times\hat{\mathbf{r}}$ (chapter 12).

**Worked example 4.** $\mathbf{a} = (3, 1, 0)$, $\mathbf{b} = (1, 2, 0)$. $x$: $a_y b_z - a_z b_y = 0 - 0 = 0$. $y$: $a_z b_x - a_x b_z = 0 - 0 = 0$. $z$: $a_x b_y - a_y b_x = 6 - 1 = 5$. So $\mathbf{a}\times\mathbf{b} = (0, 0, 5)$: perpendicular to the plane of $\mathbf{a}$ and $\mathbf{b}$, pointing along $+z$ as the right hand predicts for a counter-clockwise turn from $\mathbf{a}$ to $\mathbf{b}$, with length $5 = \sqrt{10}\sqrt5\sin45^\circ$.

**Worked example 5.** $\mathbf{a} = (1, 2, 3)$, $\mathbf{b} = (4, 5, 6)$. $x$: $2\cdot6 - 3\cdot5 = -3$. $y$: $3\cdot4 - 1\cdot6 = 6$. $z$: $1\cdot5 - 2\cdot4 = -3$. $\mathbf{a}\times\mathbf{b} = (-3, 6, -3)$. Check perpendicularity: $(-3, 6, -3)\cdot(1, 2, 3) = -3 + 12 - 9 = 0$ and $\cdot(4, 5, 6) = -12 + 30 - 18 = 0$.

**Worked example 6 (volume).** The scalar triple product $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})$ is the volume of the parallelepiped spanned by the three vectors (with a sign that records the handedness). For $\mathbf{a} = (1, 0, 0)$, $\mathbf{b} = (0, 2, 0)$, $\mathbf{c} = (0, 0, 3)$: $\mathbf{b}\times\mathbf{c} = (6, 0, 0)$ and $\mathbf{a}\cdot(6, 0, 0) = 6$, the volume of the $1\times2\times3$ box.

In [ ]:
a = np.array([3.0, 1.0, 0.0]); b = np.array([1.0, 2.0, 0.0])
print("a x b =", np.cross(a, b), "   b x a =", np.cross(b, a), "   a x a =", np.cross(a, a))
a2 = np.array([1.0, 2.0, 3.0]); b2 = np.array([4.0, 5.0, 6.0])
c2 = np.cross(a2, b2)
print("(1,2,3) x (4,5,6) =", c2, "   perpendicular checks:", c2 @ a2, c2 @ b2)
print("unit vectors: x cross y =", np.cross([1, 0, 0], [0, 1, 0]), "  y cross x =", np.cross([0, 1, 0], [1, 0, 0]))
A, B, C = np.array([1.0, 0, 0]), np.array([0, 2.0, 0]), np.array([0, 0, 3.0])
print("triple product a . (b x c) =", A @ np.cross(B, C), "   same as the determinant:", np.linalg.det(np.array([A, B, C])))

### 3.4 Position and displacement vectors

Every point in space is labelled by its position vector $\mathbf{r} = (x, y, z)$, the arrow from the origin to the point. The most important vector in electrostatics is not a position but a difference of positions: the displacement from a source point $\mathbf{r}'$ (where a charge sits) to a field point $\mathbf{r}$ (where we ask for the field),

$$\mathbf{R} = \mathbf{r} - \mathbf{r}', \qquad |\mathbf{R}| = \sqrt{(x-x')^2 + (y-y')^2 + (z-z')^2}, \qquad \hat{\mathbf{R}} = \mathbf{R}/|\mathbf{R}|.$$

Coulomb's law in chapter 2 reads $\mathbf{E} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\hat{\mathbf{R}}}{|\mathbf{R}|^2} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\mathbf{R}}{|\mathbf{R}|^3}$. The second form avoids computing the unit vector separately and is what the code in later notebooks uses.

**Worked example 7.** A charge at $\mathbf{r}' = (-1, 1, 0)$, field point $\mathbf{r} = (2, 1, 0)$. Then $\mathbf{R} = (3, 0, 0)$, $|\mathbf{R}| = 3$, $\hat{\mathbf{R}} = (1, 0, 0)$, and $\mathbf{R}/|\mathbf{R}|^3 = (3, 0, 0)/27 = (1/9, 0, 0)$. The field points away from the charge along $+x$ and its size falls as one over distance squared: at twice the distance, $\mathbf{r} = (5, 1, 0)$, we get $\mathbf{R}/|\mathbf{R}|^3 = (6, 0, 0)/216 = (1/36, 0, 0)$, a quarter as large.

In [ ]:
r_source = np.array([-1.0, 1.0, 0.0])
for r_field in (np.array([2.0, 1.0, 0.0]), np.array([5.0, 1.0, 0.0]), np.array([2.0, 4.0, 0.0])):
    R = r_field - r_source
    print(f"r = {r_field}:  R = {R}  |R| = {np.linalg.norm(R):.3f}  R_hat = {R / np.linalg.norm(R)}  R/|R|^3 = {R / np.linalg.norm(R) ** 3}")

### 3.5 Many vectors at once

Physics code rarely handles one vector. It handles thousands: the positions of all the grid points, the field at each of them. The NumPy idiom is an array of shape `(N, 3)`, one vector per row. Lengths come from `np.linalg.norm(..., axis=1)`, row-wise dot products from `np.einsum("ij,ij->i", A, B)` or `np.sum(A * B, axis=1)`, and row-wise cross products from `np.cross`. Broadcasting lets a single vector act on every row.

The cell below computes $\mathbf{R}/|\mathbf{R}|^3$ for one source and many field points in one line, which is already the core of the Coulomb superposition code in module 1.

In [ ]:
points = np.array([[2.0, 1.0, 0.0], [5.0, 1.0, 0.0], [2.0, 4.0, 0.0], [0.0, 0.0, 1.0]])      # (N, 3)
R = points - r_source                                   # broadcasting: the single source is subtracted from every row
dist = np.linalg.norm(R, axis=1)                        # (N,)
coulomb_shape = R / dist[:, None] ** 3                  # divide each row by its own |R|^3
print("distances:", dist)
print("R / |R|^3 per row:\n", coulomb_shape)
print("row-wise dot products of R with itself (= |R|^2):", np.einsum("ij,ij->i", R, R))

### 3.6 Scalar fields (section 1.2)

A function of one variable, $f(x)$, is a curve. A function of position, $f(x, y, z)$, is a scalar field: a number attached to every point in space. Temperature in a room, pressure in the atmosphere, and the electric potential $V$ in chapter 3 are all scalar fields.

Pictures. In two dimensions a scalar field can be drawn as a landscape (height $= f$) or, more usefully, as a contour map: curves along which $f$ is constant, like the height lines on a hiking map. Closely spaced contours mean steep slopes. In three dimensions the analogues are level surfaces, shells on which $f$ is constant: the equipotential surfaces of chapter 3.

To draw a field we evaluate it on a grid. The grid is every combination of a list of $x$ values with a list of $y$ values, and `np.meshgrid` builds the two arrays that hold the $x$ and $y$ coordinate of every grid point. One convention matters: with `indexing="ij"` the first array index follows $x$ and the second follows $y$, matching how we write $f_{i,j}$ on paper. The default mode swaps the two axes and silently transposes every picture. Every notebook in this repository uses `"ij"`.

In [ ]:
x = np.linspace(-2, 2, 5)
y = np.linspace(0, 1, 3)
X, Y = np.meshgrid(x, y, indexing="ij")
print("x values:", x); print("y values:", y)
print("X (one row per x value, one column per y value):\n", X)
print("Y:\n", Y)
print("grid point [i=3, j=2] sits at (x[3], y[2]) =", (X[3, 2], Y[3, 2]), "=", (x[3], y[2]))

In [ ]:
x = np.linspace(-2, 2, 81); y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y, indexing="ij")
h = x[1] - x[0]

f = X**2 / 2 + Y**2 - X * Y / 2                     # a tilted bowl
g = np.exp(-((X - 0.8)**2 + Y**2)) - np.exp(-((X + 0.8)**2 + Y**2))   # a hill next to a hollow

fig = plt.figure(figsize=(13, 9))
for k, (F, name) in enumerate(((f, "bowl f"), (g, "hill and hollow g"))):
    ax1 = fig.add_subplot(2, 2, 2 * k + 1, projection="3d")
    ax1.plot_surface(X, Y, F, cmap="viridis", alpha=0.9); ax1.set_title(name + " as a landscape"); ax1.set_xlabel("x"); ax1.set_ylabel("y")
    ax2 = fig.add_subplot(2, 2, 2 * k + 2)
    cs = ax2.contour(X, Y, F, levels=15, cmap="viridis"); ax2.clabel(cs, fontsize=7); ax2.set_aspect("equal"); ax2.set_title(name + " as a contour map")
plt.tight_layout(); plt.show()

Read the contour maps. The bowl has one minimum at the origin, closed contours around it, and contours that crowd together as you go out: the slope increases. The hill-and-hollow field has a maximum near $x = 0.8$ and a minimum near $x = -0.8$, and the contour $g = 0$ runs straight along the $y$ axis between them.

**Worked example 8.** Where is the bowl at height 1? Solve $\tfrac12x^2 + y^2 - \tfrac12xy = 1$. Along the $x$ axis ($y = 0$): $x^2 = 2$, $x = \pm1.414$. Along the $y$ axis ($x = 0$): $y = \pm1$. Along the diagonal $x = y = t$: $\tfrac12t^2 + t^2 - \tfrac12t^2 = t^2 = 1$, $t = \pm1$. So the contour $f = 1$ passes through $(\pm1.414, 0)$, $(0, \pm1)$ and $(\pm1, \pm1)$: an ellipse, tilted, as in the picture.

In [ ]:
f_func = lambda px, py: px**2 / 2 + py**2 - px * py / 2
for pt in ((1.414, 0), (0, 1), (1, 1), (-1, -1)):
    print(f"f{pt} = {f_func(*pt):.3f}")

### 3.7 Vector fields

A vector field attaches a vector to every point: wind velocity on a weather map, water velocity in a river, and the electric and magnetic fields. On paper we draw an arrow at a sample of points (a quiver plot) or we draw field lines, curves that follow the arrows everywhere (a stream plot). Arrows show magnitude by their length; field lines show magnitude by how closely they bunch.

The cell below draws six reference fields. Look at each picture and answer two questions before reading further: does anything pile up or spread out (that will be divergence), and would a small paddlewheel spin anywhere (that will be curl)? The answers for all six come in notebook 00c.

In [ ]:
x = np.linspace(-2, 2, 21); y = np.linspace(-2, 2, 21)
X, Y = np.meshgrid(x, y, indexing="ij")
reference_fields = {
    "uniform   F = (1, 0)":     (np.ones_like(X), np.zeros_like(X)),
    "radial    F = (x, y)":     (X, Y),
    "rotation  F = (-y, x)":    (-Y, X),
    "saddle    F = (x, -y)":    (X, -Y),
    "shear     F = (y, 0)":     (Y, np.zeros_like(X)),
    "sink+swirl F = (-x-y, x-y)": (-X - Y, X - Y),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, (Fx, Fy)) in zip(axes.flat, reference_fields.items()):
    ax.quiver(X, Y, Fx, Fy, pivot="mid")
    ax.set_aspect("equal"); ax.set_title(name)
plt.show()

Two of these deserve a comment now. The shear field $(y, 0)$ has all its arrows parallel, yet a paddlewheel placed in it spins: the flow is faster on one side of the wheel than the other. Parallel arrows do not mean no rotation. The sink-with-swirl field is the radial field pointed inward plus the rotation field; it does both things at once. Divergence and curl are independent properties.

Field lines are often the better picture for electric and magnetic fields. The cell draws the same fields as stream plots. Note that `streamplot` expects arrays indexed as `[y, x]`, so the `"ij"` arrays are transposed on the way in. That is the one place in these notebooks where the transpose appears, and it is deliberate.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, (Fx, Fy)) in zip(axes.flat, reference_fields.items()):
    speed = np.hypot(Fx, Fy)
    ax.streamplot(x, y, Fx.T, Fy.T, color=speed.T, cmap="viridis", density=1.2)
    ax.set_aspect("equal"); ax.set_title(name + "  (field lines, colour = speed)")
plt.show()

### 3.8 The field of a point charge as a vector field

Combine sections 3.4 and 3.7: the Coulomb field of a charge at the origin is $\mathbf{E} \propto \mathbf{r}/|\mathbf{r}|^3$. Its direction is radial and its magnitude is $1/r^2$. On a quiver plot the arrows near the charge are enormous and the far ones invisible, so it is usual to draw unit arrows and encode the magnitude as colour on a logarithmic scale. The code below does that, and the stream plot shows the familiar radial field lines.

**Worked example 9.** At $\mathbf{r} = (1, 0, 0)$ the field is $(1, 0, 0)/1 = (1, 0, 0)$ in units of $q/4\pi\epsilon_0$. At $(0, 2, 0)$ it is $(0, 2, 0)/8 = (0, 0.25, 0)$: pointing along $+y$, a quarter as strong, as $1/r^2$ demands. At $(1, 1, 0)$: $|\mathbf{r}| = \sqrt2$, $|\mathbf{r}|^3 = 2\sqrt2 = 2.828$, so $\mathbf{E} = (0.354, 0.354, 0)$ with magnitude $0.5 = 1/r^2$.

In [ ]:
x = np.linspace(-2, 2, 41); y = np.linspace(-2, 2, 41)
X, Y = np.meshgrid(x, y, indexing="ij")
Rmag = np.hypot(X, Y)
with np.errstate(divide="ignore", invalid="ignore"):
    Ex, Ey = X / Rmag**3, Y / Rmag**3
Emag = np.hypot(Ex, Ey)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
st = 2
axes[0].quiver(X[::st, ::st], Y[::st, ::st], (Ex / Emag)[::st, ::st], (Ey / Emag)[::st, ::st], np.log10(Emag)[::st, ::st], cmap="plasma", pivot="mid")
axes[0].set_aspect("equal"); axes[0].set_title("point charge: unit arrows, colour = log10 |E|")
axes[1].streamplot(x, y, Ex.T, Ey.T, color=np.log10(Emag).T, cmap="plasma", density=1.3)
axes[1].set_aspect("equal"); axes[1].set_title("field lines radiate from the charge")
plt.show()
for p in ((1.0, 0.0), (0.0, 2.0), (1.0, 1.0)):
    r = np.array(p); print(f"E at {p} = {r / np.linalg.norm(r) ** 3}   magnitude {1 / np.linalg.norm(r) ** 2:.3f}")

### Exercise: a field from a formula

Replace the body of `dipole_field` so that it returns the 2D field of two charges, $+1$ at $(0, 0.5)$ and $-1$ at $(0, -0.5)$, as the sum of two Coulomb terms $\mathbf{R}/|\mathbf{R}|^3$. The check evaluates it on the $x$ axis far away, where the field of a dipole points along $-y$ and falls as $1/r^3$ (for this dipole, $E_y \approx -1/x^3$ at large $x$).

In [ ]:
def dipole_field(X, Y):
    """Return (Ex, Ey) of a +1 charge at (0, 0.5) and a -1 charge at (0, -0.5), in units of q/(4 pi eps0)."""
    Ex = np.zeros_like(X); Ey = np.zeros_like(Y)
    for q, (sx, sy) in ((1.0, (0.0, 0.5)), (-1.0, (0.0, -0.5))):               # replace this loop with your own version
        Rx, Ry = X - sx, Y - sy
        d3 = np.hypot(Rx, Ry) ** 3
        Ex = Ex + q * Rx / d3; Ey = Ey + q * Ry / d3
    return Ex, Ey


xs = np.array([5.0, 10.0, 20.0])
Ex_far, Ey_far = dipole_field(xs, np.zeros_like(xs))
print("on the x axis:  E_y * x^3 =", Ey_far * xs**3, "  (approaches -1)   E_x =", Ex_far)
assert np.allclose(Ey_far * xs**3, -1.0, rtol=0.02) and np.allclose(Ex_far, 0.0)
Ex_d, Ey_d = dipole_field(X, Y)
plt.streamplot(x, y, Ex_d.T, Ey_d.T, color="k", density=1.3, linewidth=0.7)
plt.plot([0, 0], [0.5, -0.5], "o", color="C3"); plt.gca().set_aspect("equal"); plt.title("your dipole field"); plt.show()

## 4. Verify

Three identities that every vector calculation relies on, checked on random vectors so that they cannot pass by accident:

1. $|\mathbf{a}\times\mathbf{b}|^2 + (\mathbf{a}\cdot\mathbf{b})^2 = |\mathbf{a}|^2|\mathbf{b}|^2$ (because $\sin^2 + \cos^2 = 1$).
2. $\mathbf{a}\times(\mathbf{b}\times\mathbf{c}) = \mathbf{b}(\mathbf{a}\cdot\mathbf{c}) - \mathbf{c}(\mathbf{a}\cdot\mathbf{b})$, the "BAC minus CAB" rule, used when the magnetic vector potential appears in chapter 11.
3. $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c}) = \mathbf{b}\cdot(\mathbf{c}\times\mathbf{a}) = \mathbf{c}\cdot(\mathbf{a}\times\mathbf{b})$, cyclic symmetry of the triple product.

In [ ]:
rng = np.random.default_rng(1)
A, B, C = rng.normal(size=(3, 3))
lhs1 = np.linalg.norm(np.cross(A, B))**2 + (A @ B)**2
rhs1 = (A @ A) * (B @ B)
lhs2 = np.cross(A, np.cross(B, C)); rhs2 = B * (A @ C) - C * (A @ B)
t1, t2, t3 = A @ np.cross(B, C), B @ np.cross(C, A), C @ np.cross(A, B)
print(f"identity 1: {lhs1:.6f} = {rhs1:.6f}")
print(f"identity 2: {lhs2} = {rhs2}")
print(f"identity 3: {t1:.6f} = {t2:.6f} = {t3:.6f}")
assert np.isclose(lhs1, rhs1) and np.allclose(lhs2, rhs2) and np.isclose(t1, t2) and np.isclose(t2, t3)

## 5. Explore and rebuild

Explore. Build the field of four charges at the corners of a square, alternating in sign (a quadrupole). Draw its field lines. Then evaluate its magnitude along a diagonal at distances 5, 10, 20, 40 and find how fast it falls off: fit the exponent in $|\mathbf{E}| \propto r^{-n}$. Compare with the monopole ($n = 2$) and dipole ($n = 3$) and explain the pattern in terms of cancellation.

Rebuild. Close this notebook. On paper, write the three faces of the dot product and the component formula for the cross product, then compute worked examples 2 and 5 again by hand. In a blank editor, write a function that takes a source position, a charge, and an `(N, 3)` array of field points and returns the Coulomb field at every point in one broadcast expression.